# Ordered Logistic Regression Results for Adoption Predictors of Indigenous and Modern Knowledge in Rangeland Management Practices, Northern Kenya Exploration with `mlcroissant`
This notebook guides exploration of the FAIR^2 dataset describing socio-demographic predictors and adoption behaviors in rangeland management practices in Northern Kenya. The notebook uses the `mlcroissant` library and references entities by their `@id` throughout.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

```
https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json
```

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading

Load FAIR^2 metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json'

# Load dataset (FAIR^2) metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata.to_json()

# Display title and description
print(f"Dataset Name: {metadata['name']}")
print(f"Description: {metadata['description']}")

## 2. Data Overview

Review available record sets and fields. Reference entities using their `@id`.

Below, we enumerate all record sets present and display several records from each for context.

In [ ]:
# Collect and display record sets and their @ids
record_sets_metadata = dataset.metadata.recordSets
record_set_ids = []
for rs in record_sets_metadata:
    print(f"RecordSet @id: {rs['@id']} | Name: {rs.get('name', '<unnamed>')}")
    record_set_ids.append(rs['@id'])

# For demonstration, show a few sample records for each record set
for rsid in record_set_ids:
    print(f"\nSample records from RecordSet @id: {rsid}")
    records = list(dataset.records(record_set=rsid))
    print(records[:2])

## 3. Data Extraction

Extract records from a selected record set into a pandas DataFrame. All references use `@id`.

You can replace the selected record set id with others for further exploration.

In [ ]:
# Extract all record sets into DataFrames, referenced by their @id
dataframes = {}

for rsid in record_set_ids:
    records = list(dataset.records(record_set=rsid))
    df = pd.DataFrame(records)
    dataframes[rsid] = df

# Select a main record set for demonstration -- replace with relevant @id
example_record_set_id = record_set_ids[0] if record_set_ids else None
if example_record_set_id:
    print(f"Columns in DataFrame for RecordSet @id: {example_record_set_id}")
    print(dataframes[example_record_set_id].columns.tolist())
    display(dataframes[example_record_set_id].head())
else:
    print("No record sets found in metadata.")

## 4. Exploratory Data Analysis (EDA)

Apply common data processing: filter, normalize, group using fields referenced by `@id`.

Below, we select a numeric field for filtering, normalization, and grouping. Update the field ids as needed for your analysis.

In [ ]:
# EDA on a chosen record set and numeric field using @id
if example_record_set_id:
    df = dataframes[example_record_set_id]
    # List sample columns to choose a numeric field
    print("Available columns:", df.columns.tolist())
    
    # Replace '<numeric_field_id>' with actual column @id as needed
    numeric_field_id = df.select_dtypes(include=['number']).columns[0] if not df.select_dtypes(include=['number']).empty else None
    group_field_id = df.select_dtypes(include=['object']).columns[0] if not df.select_dtypes(include=['object']).empty else None
    threshold = 10
    if numeric_field_id:
        filtered_df = df[df[numeric_field_id] > threshold]
        print(f"Filtered records with {numeric_field_id} > {threshold}:")
        display(filtered_df.head())

        # Normalize numeric field
        filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        print(f"Normalized {numeric_field_id} for filtered records:")
        display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())
        
        # Group by object/categorical field
        if group_field_id:
            grouped_df = filtered_df.groupby(group_field_id).mean(numeric_only=True)
            print(f"Grouped data by {group_field_id}:")
            display(grouped_df.head())
        else:
            print("No object/categorical field found for grouping.")
    else:
        print("No numeric field found for EDA.")
else:
    print("Cannot perform EDA; no record set loaded.")

## 5. Visualization

Plot distribution of the normalized numeric field and demonstrate grouping relationships, referencing fields by their `@id`.

Visual explorations help understand variable distributions and group differences.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if example_record_set_id and numeric_field_id:
    # Plot distribution of normalized numeric field
    plt.figure(figsize=(8,4))
    sns.histplot(filtered_df[f"{numeric_field_id}_normalized"].dropna(), bins=25, kde=True)
    plt.title(f"Distribution of {numeric_field_id} (normalized)")
    plt.xlabel(f"{numeric_field_id}_normalized")
    plt.ylabel("Frequency")
    plt.show()

    # Grouped boxplot if group field is present
    if group_field_id:
        plt.figure(figsize=(10,4))
        sns.boxplot(x=filtered_df[group_field_id], y=filtered_df[numeric_field_id])
        plt.title(f"{numeric_field_id} by {group_field_id}")
        plt.xlabel(f"{group_field_id}")
        plt.ylabel(f"{numeric_field_id}")
        plt.show()

## 6. Conclusion

This notebook demonstrated the loading, overview, extraction, and exploratory analysis of the FAIR^2 dataset using the `mlcroissant` library.

- Entities are referenced throughout by their `@id` for clarity and consistency with the Croissant schema.
- The data presents insights into socio-demographic factors and adoption behaviors in rangeland management practices in Northern Kenya.
- Typical EDA steps, including filtering, normalization, and grouping, can be applied on individual record sets and fields.

Further analysis can be performed by selecting specific variables or models, or by exploring additional record sets in the dataset.